Import packages

In [2]:
import numpy as np
import polars as pl

Make a function that classifies peaks so it can be run on each file <br>
Classify peaks in different genomic interactions <br>
For each polyploid (`M` = majalis and `T`=traunsteineri) compare fold change (FC) and significant FDR in the comparison towards diploid parental species (`F`=fuchsii and `I`=incarnata). <br>
1. Dominant: The polyploid has a significant differential targeting (DT) towards one diploid but not the other, i.e only significant FDR in one of the diploid comparisons. We classify the peak as dominant towards the diploid without a significatn FDR.
    - If significant, check fold change and determine if it is over (positive FC) or under (negative FC) targeting
2. Transgressive: The ployploid is higher or lower than both diploids
3. Additive: The polyploid is between the diploids
    - Divide in fuchsii or incarnata high

In [74]:
def annotate_gi(data, fold_change_limit, fdr_limit):
    data = (data
            .with_columns(
                # classify dominant incarnata under targeting
                # check for a significant negative fold change towards fuchsii (MF) and no significance in incarnata (MI) 
                pl.when((pl.col("MF_logFC") < -fold_change_limit) & (pl.col("MF_FDR") <= fdr_limit) & (pl.col("MI_FDR") > fdr_limit))
                .then(pl.lit("dominant_inc_under"))
                # classify dominant incarnata  over targeting
                # check for a significant positive fold change towards fuchsii (MF) and no significance in incarnata (MI) 
                .when((pl.col("MF_logFC") > fold_change_limit) & (pl.col("MF_FDR") <= fdr_limit) & (pl.col("MI_FDR") > fdr_limit))
                .then(pl.lit("dominant_inc_over"))
                # classify dominant fuchsii under targeting
                # check for a significant negative fold change towards incarnata (MI) and no significance in fuchsii (MF)
                .when((pl.col("MI_logFC") < -fold_change_limit) & (pl.col("MI_FDR") <= fdr_limit) & (pl.col("MF_FDR") > fdr_limit))
                .then(pl.lit("dominant_fuc_under"))
                # classify dominant fuchsii over targeting
                # check for a significant positive fold change towards incarnata (MI) and no significance in fuchsii (MF)
                .when((pl.col("MI_logFC") > fold_change_limit) & (pl.col("MI_FDR") <= fdr_limit) & (pl.col("MF_FDR") > fdr_limit))
                .then(pl.lit("dominant_fuc_over"))
                # classify transgressive under
                # check for a significant negative fold change towards incarnata (MI) and fuchsii (MF)
                .when(((pl.col("MF_FDR") <= fdr_limit) & (pl.col("MI_FDR") <= fdr_limit)) & ((pl.col("MF_logFC") < -fold_change_limit) & (pl.col("MI_logFC") < -fold_change_limit)))
                .then(pl.lit("transgressive_under"))
                # classify transgressive over
                # check for a significant positive fold change towards incarnata (MI) and fuchsii (MF)
                .when(((pl.col("MF_FDR") <= fdr_limit) & (pl.col("MI_FDR") <= fdr_limit)) & ((pl.col("MF_logFC") > fold_change_limit) & (pl.col("MI_logFC") > fold_change_limit)))
                .then(pl.lit("transgressive_over"))
                # classify additive fuchsii high
                # check for a significant positive fold change towards fuchsii (MF) and a significant negative fold change towards incarnata (MI)
                .when(((pl.col("MF_FDR") <= fdr_limit) & (pl.col("MI_FDR") <= fdr_limit)) & ((pl.col("MF_logFC") > fold_change_limit) & (pl.col("MI_logFC") < -fold_change_limit)))
                .then(pl.lit("additive_high_fuc"))
                # classify additive incarnats high
                # check for a significant negative fold change towards fuchsii (MF) and a significant positive fold change towards incarnata (MI)
                .when(((pl.col("MF_FDR") <= fdr_limit) & (pl.col("MI_FDR") <= fdr_limit)) & ((pl.col("MF_logFC") < -fold_change_limit) & (pl.col("MI_logFC") > fold_change_limit)))
                .then(pl.lit("additive_high_inc"))
                # if no argument is met, assign no_classification
                .otherwise(pl.lit("no_classification"))
                # name the column gi = genomic interaction, maj = majalis
                .alias("gi_maj")
                          )
            .with_columns(
                # classify dominant incarnata under targeting
                # check for a significant negative fold change towards fuchsii (TF) and no significance in incarnata (TI) 
                pl.when((pl.col("TF_logFC") < -fold_change_limit) & (pl.col("TF_FDR") <= fdr_limit) & (pl.col("TI_FDR") > fdr_limit))
                .then(pl.lit("dominant_inc_under"))
                # classify dominant incarnata  over targeting
                # check for a significant positive fold change towards fuchsii (TF) and no significance in incarnata (TI) 
                .when((pl.col("TF_logFC") > fold_change_limit) & (pl.col("TF_FDR") <= fdr_limit) & (pl.col("TI_FDR") > fdr_limit))
                .then(pl.lit("dominant_inc_over"))
                # classify dominant fuchsii under targeting
                # check for a significant negative fold change towards incarnata (TI) and no significance in fuchsii (TF)
                .when((pl.col("TI_logFC") < -fold_change_limit) & (pl.col("TI_FDR") <= fdr_limit) & (pl.col("TF_FDR") > fdr_limit))
                .then(pl.lit("dominant_fuc_under"))
                # classify dominant fuchsii over targeting
                # check for a significant positive fold change towards incarnata (TI) and no significance in fuchsii (TF)
                .when((pl.col("TI_logFC") > fold_change_limit) & (pl.col("TI_FDR") <= fdr_limit) & (pl.col("TF_FDR") > fdr_limit))
                .then(pl.lit("dominant_fuc_over"))
                # classify transgressive under
                # check for a significant negative fold change towards incarnata (TI) and fuchsii (TF)
                .when(((pl.col("TF_FDR") <= fdr_limit) & (pl.col("TI_FDR") <= fdr_limit)) & ((pl.col("TF_logFC") < -fold_change_limit) & (pl.col("TI_logFC") < -fold_change_limit)))
                .then(pl.lit("transgressive_under"))
                # classify transgressive over
                # check for a significant positive fold change towards incarnata (TI) and fuchsii (TF)
                .when(((pl.col("TF_FDR") <= fdr_limit) & (pl.col("TI_FDR") <= fdr_limit)) & ((pl.col("TF_logFC") > fold_change_limit) & (pl.col("TI_logFC") > fold_change_limit)))
                .then(pl.lit("transgressive_over"))
                # classify additive fuchsii high
                # check for a significant positive fold change towards fuchsii (TF) and a significant negative fold change towards incarnata (TI)
                .when(((pl.col("TF_FDR") <= fdr_limit) & (pl.col("TI_FDR") <= fdr_limit)) & ((pl.col("TF_logFC") > fold_change_limit) & (pl.col("TI_logFC") < -fold_change_limit)))
                .then(pl.lit("additive_high_fuc"))
                # classify additive incarnats high
                # check for a significant negative fold change towards fuchsii (TF) and a significant positive fold change towards incarnata (TI)
                .when(((pl.col("TF_FDR") <= fdr_limit) & (pl.col("TI_FDR") <= fdr_limit)) & ((pl.col("TF_logFC") < -fold_change_limit) & (pl.col("TI_logFC") > fold_change_limit)))
                .then(pl.lit("additive_high_inc"))
                # if no argument is met, assign no_classification
                .otherwise(pl.lit("no_classification"))
                # name the column gi = genomic interaction, tra = traunsteineri
                .alias("gi_tra")
               )
            .with_columns([pl.col(x).round(2) for x in ["MF_logFC", "MI_logFC", "TF_logFC", "TI_logFC"]])
            .select(["peakID", "MF_logFC", "MF_FDR", "MI_logFC", "MI_FDR", "gi_maj", "TF_logFC", "TF_FDR", "TI_logFC", "TI_FDR", "gi_tra"])
            .sort("peakID")
            )
    return data

Set fold change limit to be considered, we use 1.5 <br>
Set FDR threshold to 0.05

In [75]:
fc_lim = 1.5
fdr_lim = 0.05

Classify peaks in the 20-24 nt comparisons (gene focused)

In [ ]:
gene_4x2x_20_24 = pl.read_csv("../../02_DT-analysis/4x-vs-2x_20to24nt.txt", separator="\t")
gene_4x2x_20_24 = annotate_gi(gene_4x2x_20_24, fc_lim, fdr_lim)
gene_4x2x_20_24.write_csv("../../03_genomic-intercations/all-DTpeaks_4x-vs-2x_20to24nt.txt", 
                     separator="\t")

Classify peaks in the 20-23 nt comparisons (TE focused)

In [68]:
te_4x2x_20_23 = pl.read_csv("../../02_DT-analysis/4x-vs-2x_20to23nt.txt", separator="\t")
te_4x2x_20_23 = annotate_gi(te_4x2x_20_23, fc_lim, fdr_lim)
te_4x2x_20_23.write_csv("../../03_genomic-intercations/all-DTpeaks_4x-vs-2x_20to23nt.txt", 
                     separator="\t")

Classify peaks in the 24 nt comparisons (TE focused)

In [70]:
te_4x2x_24 = pl.read_csv("../../02_DT-analysis/4x-vs-2x_24nt.txt", separator="\t")
te_4x2x_24 = annotate_gi(te_4x2x_24, fc_lim, fdr_lim)
te_4x2x_24.write_csv("../../03_genomic-intercations/all-DTpeaks_4x-vs-2x_24nt.txt", 
                     separator="\t")